In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/your folder"

Mounted at /content/drive


In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, confusion_matrix

SEEDS = (42, 43, 44)
NBOOT = 500                                   # use 200 for a quick first pass
ddr_idx = np.sort(pd.read_csv(BASE_PATH + "/ddr_clean_split.csv").csv_row.values)   # rows kept after de-duplication

FAM = {   # name: (APTOS test file, IDRiD file, DDR file, DDR file holds all 12,522 rows?)
 "VQC":            ("test_preds_v3_circuit_aNone_pe0_L4_s{s}.npz", "ext_idrid_v3_circuit_aNone_pe0_L4_s{s}.npz", "ext_ddr_all_v3_circuit_aNone_pe0_L4_s{s}.npz", True),
 "no-VQC":         ("test_preds_skip50_skip_aNone_pe0_L4_s{s}.npz", "ext_idrid_skip50_skip_aNone_pe0_L4_s{s}.npz", "ext_ddr_all_skip50_skip_aNone_pe0_L4_s{s}.npz", True),
 "classical-only": ("test_preds_ctrl_classical_L4_s{s}.npz", "ext_idrid_ctrl_classical_L4_s{s}.npz", "ext_ddrclean_ctrl_classical_L4_s{s}.npz", False),
 "cnn_ms":         ("test_preds_ctrl_cnn_ms_L4_s{s}.npz", "ext_idrid_ctrl_cnn_ms_L4_s{s}.npz", "ext_ddrclean_ctrl_cnn_ms_L4_s{s}.npz", False),
 "cnn_single":     ("test_preds_ctrl_cnn_single_L4_s{s}.npz", "ext_idrid_ctrl_cnn_single_L4_s{s}.npz", "ext_ddrclean_ctrl_cnn_single_L4_s{s}.npz", False),
}
DS = ["APTOS test", "IDRiD", "DDR dedup"]

def load(fam, k, s):
    f = FAM[fam][k].format(s=s)
    d = np.load(f"{BASE_PATH}/{f}")
    P, Y = d["P"], d["Y"]
    if k == 2 and FAM[fam][3]: P, Y = P[ddr_idx], Y[ddr_idx]
    return P, Y

L = [0, 1, 2, 3, 4]
def metrics(P, Y):
    pr = P.argmax(1); cm = confusion_matrix(Y, pr, labels=L)
    return dict(acc=accuracy_score(Y, pr), f1=f1_score(Y, pr, average="macro", labels=L, zero_division=0),
                qwk=cohen_kappa_score(Y, pr, weights="quadratic", labels=L), auc=roc_auc_score(Y, P, multi_class="ovr", labels=L),
                ref_auc=roc_auc_score((Y >= 2).astype(int), P[:, 2:].sum(1)),
                sens=cm[2:, 2:].sum() / cm[2:].sum(), spec=cm[:2, :2].sum() / cm[:2].sum())

# ---- table: mean +- SD over seeds, per dataset and family ----
rows = []
for k, ds in enumerate(DS):
    for fam in FAM:
        try: ms = [metrics(*load(fam, k, s)) for s in SEEDS]
        except FileNotFoundError as e: print("missing:", e); continue
        r = dict(dataset=ds, family=fam, seeds=len(ms))
        for m in ms[0]:
            v = np.array([x[m] for x in ms]); r[m] = f"{v.mean():.3f} ± {v.std(ddof=1):.3f}"
        rows.append(r)
tab = pd.DataFrame(rows); print(tab.to_string(index=False))
tab.to_csv(BASE_PATH + "/summary_all_families.csv", index=False)

# ---- paired bootstrap: each family minus the classical-only model, 3-seed mean, test images resampled ----
REF = "classical-only"; rng = np.random.default_rng(0); out = []
for k, ds in enumerate(DS):
    try: ref = [load(REF, k, s) for s in SEEDS]
    except FileNotFoundError: continue
    n = len(ref[0][1]); idx = [rng.integers(0, n, n) for _ in range(NBOOT)]
    def boot(fam):
        P = [load(fam, k, s) for s in SEEDS]
        d = {"acc": [], "qwk": [], "ref_auc": []}
        for i in idx:
            a = [metrics(p[i], y[i]) for p, y in ref]; b = [metrics(p[i], y[i]) for p, y in P]
            for m in d: d[m].append(np.mean([x[m] for x in b]) - np.mean([x[m] for x in a]))
        return d
    for fam in FAM:
        if fam == REF: continue
        try: d = boot(fam)
        except FileNotFoundError: continue
        for m, v in d.items():
            lo, hi = np.percentile(v, [2.5, 97.5])
            out.append(dict(dataset=ds, comparison=f"{fam} - {REF}", metric=m, diff=round(float(np.mean(v)), 3), ci95=f"[{lo:.3f}, {hi:.3f}]", includes_zero=bool(lo <= 0 <= hi)))
print(pd.DataFrame(out).to_string(index=False))
pd.DataFrame(out).to_csv(BASE_PATH + "/bootstrap_vs_classical.csv", index=False)

   dataset         family  seeds           acc            f1           qwk           auc       ref_auc          sens          spec
APTOS test            VQC      3 0.841 ± 0.006 0.612 ± 0.009 0.888 ± 0.012 0.931 ± 0.016 0.970 ± 0.014 0.950 ± 0.037 0.927 ± 0.019
APTOS test         no-VQC      3 0.838 ± 0.008 0.598 ± 0.048 0.893 ± 0.014 0.924 ± 0.034 0.967 ± 0.022 0.968 ± 0.018 0.913 ± 0.027
APTOS test classical-only      3 0.851 ± 0.007 0.643 ± 0.006 0.900 ± 0.009 0.935 ± 0.007 0.972 ± 0.007 0.930 ± 0.000 0.933 ± 0.008
APTOS test         cnn_ms      3 0.848 ± 0.010 0.641 ± 0.010 0.873 ± 0.013 0.898 ± 0.007 0.972 ± 0.003 0.904 ± 0.040 0.938 ± 0.004
APTOS test     cnn_single      3 0.853 ± 0.000 0.640 ± 0.006 0.878 ± 0.015 0.892 ± 0.007 0.973 ± 0.005 0.930 ± 0.009 0.938 ± 0.000
     IDRiD            VQC      3 0.481 ± 0.019 0.427 ± 0.010 0.647 ± 0.067 0.796 ± 0.023 0.919 ± 0.014 0.869 ± 0.054 0.819 ± 0.153
     IDRiD         no-VQC      3 0.467 ± 0.013 0.394 ± 0.034 0.616 ± 0.010 0.797 ± 